# Séance 1 — Le problème de la fraude et les données simulées
*Session 1 — The fraud problem and simulated data*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/01_contexte_simulation.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ● séance complète |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ● séance complète |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | ● séance complète |

**À la fin de cette séance, vous saurez :**

1. décrire le cycle de vie d'une transaction carte et les principales typologies de fraude ;
2. expliquer pourquoi l'exactitude (accuracy) est trompeuse sur des classes très déséquilibrées ;
3. mesurer le délai d'étiquetage et ses conséquences sur l'entraînement d'un modèle ;
4. situer les 16 étapes du pipeline Fraude-360 dans une architecture en médaillon.

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

## 1. Contexte : une coopérative financière fictive

Une coopérative de services financiers reçoit chaque jour des milliers de transactions par carte. Environ **0,45 %**
sont frauduleuses. Chaque fraude manquée coûte le **montant de la transaction**; chaque fausse alerte coûte une **revue manuelle**
(environ 12 $) et un membre mécontent. L'équipe d'enquête ne peut traiter qu'une vingtaine d'alertes par jour.

Le projet **Fraude-360** reproduit, sur des données 100 % synthétiques, toute la chaîne d'une équipe d'analytique :

| Couche | Rôle | Séances |
|---|---|---|
| Atterrissage → **Bronze** | fichiers bruts, copie fidèle + métadonnées | 1–2 |
| **Silver** | nettoyage, dédoublonnage, contrôles qualité | 3 |
| **Gold** | variables comportementales (vélocité, géographie, réputation) | 4–5 |
| Modèles | entraînement, calibration, seuil économique, registre | 6–8 |
| Production | test A/B, API de scoring, surveillance, réentraînement | 9–12 |
| Gouvernance & valeur | équité, fiche de modèle, crédit IFRS 9, Power BI, IA générative | 13–14 |

Les typologies simulées : carte perdue/volée, achat en ligne sans carte (CNP), usurpation de compte, test de carte,
et — plus tard dans le cours — une **nouvelle** arnaque au virement instantané.

## 2. Générer les données

In [ ]:
cfg = preparer("generer")          # simule 60 jours de transactions (≈ 60 000 lignes)
fichiers = sorted(cfg.chemin("atterrissage").glob("*.csv"))
print(len(fichiers), "fichiers quotidiens, par exemple :", fichiers[0].name)
brut = pd.concat([pd.read_csv(f, dtype=str) for f in fichiers], ignore_index=True)
print(brut.shape)
brut.head()

Toutes les colonnes sont lues comme du **texte** : c'est volontaire. Les systèmes sources envoient parfois des montants
avec une virgule (`12,50`), des devises en minuscules ou des doublons. Nous les traiterons à la séance 3.

### Exercice 1.1 — 🟢 Tous les parcours

Calculez `taux_fraude`, la proportion de transactions frauduleuses (`est_fraude == 1`; une valeur manquante
compte comme 0), puis `repartition`, la part de chaque `typologie_fraude` **parmi les fraudes** (une `Series` qui somme à 1).

In [ ]:
# VOTRE CODE ICI
taux_fraude = ...
repartition = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_y = pd.to_numeric(brut["est_fraude"], errors="coerce").fillna(0)
verifier_proche(taux_fraude, (_y == 1).mean(), 1e-9)
verifier_proche(repartition.sum(), 1.0, 1e-9, "La répartition doit sommer à 1")
verifier(0.002 < taux_fraude < 0.01, "Le taux de fraude devrait être d'environ 0,45 %")

### Exercice 1.2 — 🟢 Tous les parcours

Un « modèle » paresseux prédit **toujours** « légitime ». Calculez son exactitude `exactitude_naive`, et le
`montant_fraude_total` (somme des montants frauduleux, en dollars) qu'il laisserait passer. Astuce : les montants sont du texte;
remplacez la virgule par un point avant `pd.to_numeric`.

In [ ]:
# VOTRE CODE ICI
exactitude_naive = ...
montant_fraude_total = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(exactitude_naive > 0.99, "Le modèle naïf a une exactitude supérieure à 99 % !")
_m = pd.to_numeric(brut["montant"].str.strip().str.replace(",", ".", regex=False), errors="coerce")
verifier_proche(montant_fraude_total, _m[pd.to_numeric(brut.est_fraude, errors="coerce").fillna(0) == 1].sum(), 1e-6)

**❓ Question (🔶 Gestion — décision d'affaires) —** Quelle(s) mesure(s) présenteriez-vous à la direction à la place de l'exactitude ? Pensez au rappel, à la précision et surtout aux **dollars** évités et aux coûts de revue.

*Votre réponse ici.*

### Exercice 1.3 — 🟣 Gestion + Informatique

La fraude n'est confirmée qu'après la contestation du membre (`horodatage_label`). Calculez `delai_jours`,
une `Series` du délai (en jours, décimales permises) entre `horodatage` et `horodatage_label` pour les fraudes, puis affichez son
histogramme.

In [ ]:
# VOTRE CODE ICI
delai_jours = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(len(delai_jours) > 50, "Calculez le délai pour toutes les fraudes étiquetées")
verifier(4 < delai_jours.mean() < 10, "Le délai moyen simulé est d'environ 7 jours")

**❓ Question (🔷 Informatique — approfondissement) —** Pourquoi ne peut-on pas utiliser les 7 derniers jours de données pour entraîner un modèle, ni pour mesurer son rappel ?

*Votre réponse ici.*

## Bilan

- La fraude est rare (≈ 0,45 %) : l'exactitude ne veut rien dire, on raisonne en rappel, précision et dollars.
- Les étiquettes arrivent avec retard : le temps structure tout le projet (découpage, surveillance).
- Les données brutes sont imparfaites : la qualité sera traitée explicitement (séance 3).

**Pour aller plus loin :**

- `src/fraude360/generation/simulateur.py` : comment chaque typologie est simulée.
- Rapport ACFE *Occupational Fraud* et statistiques du Centre antifraude du Canada.

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*